# Chapter 3 — Introduction to Neural Prediction: Forward Propagation

**Mata Kuliah:** Deep Learning
**Buku:** Grokking Deep Learning (Andrew W. Trask, Manning)
**Chapter 3:** Introduction to Neural Prediction: Forward Propagation

Notebook ini berisi reproduksi kode dari buku (diambil dari repository
resmi penulis) beserta penjelasan konsep dan rangkuman chapter.

## Pendahuluan

Chapter ini membahas langkah pertama dari siklus **predict, compare,
learn**, yaitu **predict**. Anda membangun neural network pertama dan
memakainya untuk memprediksi. Inti chapter: *untuk memprediksi, neural
network melakukan weighted sum (jumlah berbobot) berulang atas input.*

Contoh sepanjang chapter: memprediksi apakah tim baseball menang
(`win?`) dari data seperti rata-rata jumlah jari kaki pemain
(`toes`), persentase kemenangan (`wlrec`), dan jumlah penggemar
(`nfans`).

# A Simple Neural Network Making a Prediction

**Neural network paling sederhana:** satu input, satu *weight*, satu
prediksi, yaitu `prediction = input * weight`. Weight adalah "tombol
volume": weight besar memperkuat input, weight kecil memperlemahnya.
Jaringan hanya mengetahui input yang diberikan saat itu dan tidak
mengingat prediksi sebelumnya.

### What is a Neural Network?

Neural network, pada bentuk paling dasar, adalah satu atau lebih weight
yang dikalikan dengan data input untuk membuat prediksi. *Input data*
adalah angka yang direkam dari dunia nyata (mudah diketahui), dan
*prediction* adalah apa yang dikatakan jaringan. Prediksi tidak selalu
benar, tetapi jaringan bisa belajar dari kesalahan dengan menaikkan atau
menurunkan weight.

In [ ]:
# The network:

weight = 0.1 
def neural_network(input, weight):
    prediction = input * weight
    return prediction

# How we use the network to predict something:

number_of_toes = [8.5, 9.5, 10, 9]
input = number_of_toes[0]
pred = neural_network(input,weight)
print(pred)

# Making a Prediction with Multiple Inputs

**Banyak input:** jaringan dapat menerima beberapa data sekaligus. Setiap
input dikalikan weight miliknya sendiri, lalu hasilnya dijumlahkan. Ini
disebut **weighted sum** atau **dot product**. Input dan weights
berbentuk **vektor** (daftar angka). Operasi yang memasangkan elemen
berdasarkan posisi disebut *elementwise*.

Intuisi dot product: ukuran **kemiripan** dua vektor. Weight positif
berperan seperti AND/OR, weight negatif seperti NOT.

### Complete Runnable Code

Berikut kode lengkap dengan Python murni (list dan angka). Versi NumPy di
bawahnya lebih ringkas dan akan dipakai seterusnya.

In [ ]:
def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, 0] 
    
def neural_network(input, weights):
    pred = w_sum(input,weights)
    return pred

# This dataset is the current
# status at the beginning of
# each game for the first 4 games
# in a season.

# toes = current number of toes
# wlrec = current games won (percent)
# nfans = fan count (in millions)

toes =  [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [toes[0],wlrec[0],nfans[0]]
pred = neural_network(input,weights)

print(pred)

### NumPy Code

**NumPy:** `np.dot(a, b)` menghitung weighted sum dengan sangat cepat dan
ringkas. Mulai chapter ini dan seterusnya, NumPy menjadi satu-satunya
library matriks yang dipakai.

In [ ]:
import numpy as np
weights = np.array([0.1, 0.2, 0])
def neural_network(input, weights):
    pred = input.dot(weights)
    return pred
    
toes =  np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65, 0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

# Input corresponds to every entry
# for the first game of the season.

input = np.array([toes[0],wlrec[0],nfans[0]])
pred = neural_network(input,weights)

print(pred)

# Making a Prediction with Multiple Outputs

**Banyak output:** satu input dapat dipakai untuk beberapa prediksi
sekaligus. Setiap output memiliki weight sendiri; hasilnya adalah vektor
prediksi (misal peluang menang, sedih, dan lain-lain). Prosesnya sama
seperti kasus satu input, diulang untuk tiap output.

In [ ]:
# Instead of predicting just 
# whether the team won or lost, 
# now we're also predicting whether
# they are happy/sad AND the percentage
# of the team that is hurt. We are
# making this prediction using only
# the current win/loss record.

def ele_mul(number,vector):
    output = [0,0,0]
    assert(len(output) == len(vector))
    for i in range(len(vector)):
        output[i] = number * vector[i]
    return output

weights = [0.3, 0.2, 0.9] 

def neural_network(input, weights):
    pred = ele_mul(input,weights)
    return pred
    
wlrec = [0.65, 0.8, 0.8, 0.9]
input = wlrec[0]
pred = neural_network(input,weights)

print(pred)

# Predicting with Multiple Inputs & Outputs

**Banyak input dan banyak output:** gabungan keduanya. Weights menjadi
**matriks** (satu baris vektor weight per output). Setiap output dihitung
sebagai dot product dari seluruh input dengan satu baris weights. Fungsi
`vect_mat_mul` pada kode melakukan hal ini.

In [ ]:
            #toes %win #fans
weights = [ [0.1, 0.1, -0.3], #hurt?
            [0.1, 0.2, 0.0], #win?
            [0.0, 1.3, 0.1] ] #sad?

def w_sum(a,b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

def vect_mat_mul(vect,matrix):
    assert(len(vect) == len(matrix))
    output = [0,0,0]
    for i in range(len(vect)):
        output[i] = w_sum(vect,matrix[i])
    return output

def neural_network(input, weights):
    pred = vect_mat_mul(input,weights)
    return pred

# This dataset is the current
# status at the beginning of
# each game for the first 4 games
# in a season.

# toes = current number of toes
# wlrec = current games won (percent)
# nfans = fan count (in millions)

toes =  [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65,0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [toes[0],wlrec[0],nfans[0]]
pred = neural_network(input,weights)

print(pred)

# Predicting on Predictions

**Jaringan bertumpuk (stacked):** keluaran satu jaringan dapat menjadi
input jaringan berikutnya. Hasilnya adalah jaringan dengan **hidden
layer**, yaitu lapisan di antara input dan output. Jaringan seperti ini
menjadi dasar *deep* neural network.

In [ ]:
            #toes %win #fans
ih_wgt = [ [0.1, 0.2, -0.1], #hid[0]
           [-0.1,0.1, 0.9], #hid[1]
           [0.1, 0.4, 0.1] ] #hid[2]

           #hid[0] hid[1] hid[2]
hp_wgt = [ [0.3, 1.1, -0.3], #hurt?
           [0.1, 0.2, 0.0], #win?
           [0.0, 1.3, 0.1] ] #sad?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):
    hid = vect_mat_mul(input,weights[0])
    pred = vect_mat_mul(hid,weights[1])
    return pred

toes =  [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65,0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

# Input corresponds to every entry
# for the first game of the season.

input = [toes[0],wlrec[0],nfans[0]]
pred = neural_network(input,weights)

print(pred)

# NumPy Version

Versi NumPy dari jaringan bertumpuk: `a.dot(b)` untuk vektor dan matriks.
Perhatikan urutan operasi dan bentuk (*shape*) array.

In [ ]:
import numpy as np

#toes %win #fans
ih_wgt = np.array([ 
            [0.1, 0.2, -0.1], #hid[0]
            [-0.1,0.1, 0.9], #hid[1]
            [0.1, 0.4, 0.1]]).T #hid[2]


# hid[0] hid[1] hid[2]
hp_wgt = np.array([  
            [0.3, 1.1, -0.3], #hurt?
            [0.1, 0.2, 0.0], #win?
            [0.0, 1.3, 0.1] ]).T #sad?

weights = [ih_wgt, hp_wgt]

def neural_network(input, weights):

    hid = input.dot(weights[0])
    pred = hid.dot(weights[1])
    return pred


toes =  np.array([8.5, 9.5, 9.9, 9.0])
wlrec = np.array([0.65,0.8, 0.8, 0.9])
nfans = np.array([1.2, 1.3, 0.5, 1.0])

input = np.array([toes[0],wlrec[0],nfans[0]])

pred = neural_network(input,weights)
print(pred)

# A Quick Primer on NumPy

**Primer NumPy:** vektor dan matriks dibuat dengan `np.array`. Operasi `+`,
`*` bersifat elementwise. Perkalian vektor dengan matriks memakai `.dot`.
Bentuk (*shape*) matriks menentukan apakah perkalian valid: jumlah kolom
pertama harus sama dengan jumlah baris kedua.

In [ ]:
import numpy as np

a = np.array([0,1,2,3]) # a vector
b = np.array([4,5,6,7]) # another vector
c = np.array([[0,1,2,3], # a matrix
              [4,5,6,7]])

d = np.zeros((2,4)) # (2x4 matrix of zeros)
e = np.random.rand(2,5) # random 2x5
# matrix with all numbers between 0 and 1

print(a)
print(b)
print(c)
print(d)
print(e)

In [ ]:
print(a * 0.1) # multiplies every number in vector "a" by 0.1
      
print(c * 0.2) # multiplies every number in matrix "c" by 0.2
      
print(a * b) # multiplies elementwise between a and b (columns paired up)
      
print(a * b * 0.2) # elementwise multiplication then multiplied by 0.2
      
print(a * c) # since c has the same number of columns as a, this performs
# elementwise multiplication on every row of the matrix "c"

try:
    print(a * e)
except ValueError as err:
    print('ValueError:', err)
# since a and e don't have the same number of columns, this
# throws a "Value Error: operands could not be broadcast together with.."

In [ ]:
a = np.zeros((1,4)) # vector of length 4
b = np.zeros((4,3)) # matrix with 4 rows & 3 columns

c = a.dot(b)
print(c.shape)

In [ ]:
a = np.zeros((2,4)) # matrix with 2 rows and 4 columns
b = np.zeros((4,3)) # matrix with 4 rows & 3 columns

c = a.dot(b)
print(c.shape) # outputs (2,3)

e = np.zeros((2,1)) # matrix with 2 rows and 1 columns
f = np.zeros((1,3)) # matrix with 1 row & 3 columns

g = e.dot(f)
print(g.shape) # outputs (2,3)

h = np.zeros((5,4)).T # matrix with 4 rows and 5 columns
i = np.zeros((5,6)) # matrix with 6 rows & 5 columns

j = h.dot(i)
print(j.shape) # outputs (4,6)

h = np.zeros((5,4)) # matrix with 5 rows and 4 columns
i = np.zeros((5,6)) # matrix with 5 rows & 6 columns
try:
    j = h.dot(i)
    print(j.shape)
except ValueError as err:
    print('ValueError:', err)  # throws an error

## Key Ideas

- Neural network = weights yang dikalikan dengan input untuk memprediksi.
- Weighted sum (dot product) mengukur kemiripan antara input dan weights.
- Weights dapat berbentuk skalar, vektor, atau matriks sesuai jumlah input dan output.
- Jaringan bertumpuk memakai output satu layer sebagai input layer berikutnya.
- Forward propagation = mempropagasikan aktivasi maju untuk membuat prediksi.

## Kesimpulan


Chapter 3 memperkenalkan **forward propagation**: jaringan menerima
input dan menghasilkan prediksi dengan mempropagasikan *aktivasi*
(semua angka selain weights, yang unik pada tiap prediksi) maju melalui
jaringan. Seluruh arsitektur, dari satu weight sampai jaringan bertumpuk
dengan banyak input dan output, dibangun dari beberapa aturan sederhana
yang diulang: perkalian dengan weight dan penjumlahan (dot product).
**Kecerdasan jaringan berada pada nilai weights.** Chapter 4 akan
menjelaskan cara menentukan weights agar prediksi akurat.